# Exploración inicial del dataset Olist

Primera etapa del proyecto **olist-data-pipeline**.

El objetivo de este notebook es **entender los datos antes de escribir el pipeline**:
qué contiene cada archivo, en qué estado llega, cómo se relacionan entre sí y con qué
granularidad está registrada la información.

Se trabaja con dos archivos del *Brazilian E-Commerce Public Dataset by Olist*, ubicados
en la carpeta `data/`:

| Archivo | Contenido |
|---|---|
| `olist_orders_dataset.csv` | Un registro por pedido (cabecera del pedido) |
| `olist_order_items_dataset.csv` | Detalle de los ítems que componen cada pedido |

## 1. Librerías y configuración

Se importa `pandas` para el análisis y `pathlib` para manejar las rutas de forma
independiente del sistema operativo.

In [1]:
import pandas as pd
from pathlib import Path

# Que no se recorten columnas al mostrar los DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("pandas:", pd.__version__)

pandas: 3.0.5


## 2. Carga de los datos

Los CSV se leen desde `data/bronze/`, la capa Bronze del pipeline. El notebook vive en `notebooks/`, por eso la ruta sube un
nivel; el `if` permite además ejecutarlo desde la raíz del proyecto.

In [2]:
DATA_DIR = Path("..") / "data" / "bronze"
if not DATA_DIR.exists():
    DATA_DIR = Path("data") / "bronze"

orders_df = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items_df = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

print("orders_df      ->", orders_df.shape)
print("order_items_df ->", order_items_df.shape)

orders_df      -> (99441, 8)
order_items_df -> (112650, 7)


---
## 3. Exploración de `orders_df`

Tabla de cabecera de pedidos. Las columnas de interés son `order_id`, `customer_id`,
`order_status` y las cinco columnas de fechas del ciclo de vida del pedido.

### 3.1 Primeras filas

In [3]:
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### 3.2 Tamaño y columnas

In [4]:
print(f"Registros: {len(orders_df):,}")
print(f"Columnas : {orders_df.shape[1]}")

Registros: 99,441
Columnas : 8


In [5]:
list(orders_df.columns)

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

### 3.3 Tipos de datos

`info()` muestra el tipo de cada columna y cuántos valores no nulos tiene. Las fechas
llegan como texto porque `read_csv` no las interpreta automáticamente.

In [6]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


### 3.4 Conversión de las columnas de fecha

Se convierten a `datetime` para poder calcular rangos y ordenar cronológicamente.
`errors="coerce"` transforma en `NaT` cualquier valor que no se pueda interpretar, en
lugar de cortar la ejecución.

In [7]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col], errors="coerce")

orders_df[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### 3.5 Estadísticas descriptivas

Con `include="all"` se resumen también las columnas de texto (cantidad de valores
únicos, valor más frecuente) y las de fecha.

In [8]:
orders_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1,NaN,NaN,NaN,NaN,NaN,NaN
order_status,99441,8,delivered,96478,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,99441,NaN,NaN,NaN,2017-12-31 08:43:12.776581,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18
order_approved_at,99281,NaN,NaN,NaN,2017-12-31 18:35:24.098800,2016-09-15 12:16:38,2017-09-12 23:24:16,2018-01-19 11:36:13,2018-05-04 20:35:10,2018-09-03 17:40:06
order_delivered_carrier_date,97658,NaN,NaN,NaN,2018-01-04 21:49:48.138278,2016-10-08 10:34:01,2017-09-15 22:28:50.250000,2018-01-24 16:10:58,2018-05-08 13:37:45,2018-09-11 19:48:28
order_delivered_customer_date,96476,NaN,NaN,NaN,2018-01-14 12:09:19.035542,2016-10-11 13:46:32,2017-09-25 22:07:22.250000,2018-02-02 19:28:10.500000,2018-05-15 22:48:52.250000,2018-10-17 13:22:46
order_estimated_delivery_date,99441,NaN,NaN,NaN,2018-01-24 03:08:37.730111,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00


### 3.6 Cantidad de pedidos e identificadores únicos

Interesa confirmar si `order_id` es realmente la clave primaria de esta tabla, es decir,
si hay exactamente una fila por pedido.

In [9]:
print("Filas totales         :", f"{len(orders_df):,}")
print("order_id únicos       :", f"{orders_df['order_id'].nunique():,}")
print("customer_id únicos    :", f"{orders_df['customer_id'].nunique():,}")
print("¿Una fila por pedido? :", len(orders_df) == orders_df["order_id"].nunique())

Filas totales         : 99,441
order_id únicos       : 99,441
customer_id únicos    : 99,441
¿Una fila por pedido? : True


### 3.7 Distribución de los estados del pedido

`order_status` indica en qué punto del ciclo se encuentra cada pedido. Es clave para el
pipeline: probablemente haya que filtrar por los pedidos efectivamente concretados.

In [10]:
pd.DataFrame(
    {
        "cantidad": orders_df["order_status"].value_counts(),
        "porcentaje": orders_df["order_status"].value_counts(normalize=True).mul(100).round(2),
    }
)

,cantidad,porcentaje
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


### 3.8 Rango de fechas del dataset

Mínimo, máximo y cantidad de nulos de cada columna de fecha. Los nulos son esperables:
un pedido cancelado nunca llega a tener fecha de entrega.

In [11]:
pd.DataFrame(
    {
        "minimo": orders_df[date_cols].min(),
        "maximo": orders_df[date_cols].max(),
        "nulos": orders_df[date_cols].isna().sum(),
    }
)

,minimo,maximo,nulos
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0


Distribución de los pedidos a lo largo del tiempo, agrupados por mes de compra.

In [12]:
pedidos_por_mes = (
    orders_df["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()
)

print("Primer mes:", pedidos_por_mes.index.min())
print("Último mes:", pedidos_por_mes.index.max())
pedidos_por_mes

Primer mes: 2016-09
Último mes: 2018-10


order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

---
## 4. Exploración de `order_items_df`

Tabla de detalle. Las columnas de interés son `order_id`, `order_item_id`, `product_id`,
`seller_id`, `price` y `freight_value`.

### 4.1 Primeras filas

In [13]:
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### 4.2 Tamaño y columnas

In [14]:
print(f"Registros: {len(order_items_df):,}")
print(f"Columnas : {order_items_df.shape[1]}")

Registros: 112,650
Columnas : 7


In [15]:
list(order_items_df.columns)

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

### 4.3 Tipos de datos

También se convierte `shipping_limit_date` a `datetime`.

In [16]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [17]:
order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"], errors="coerce"
)

order_items_df.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### 4.4 Estadísticas descriptivas

In [18]:
order_items_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,112650,98666,8272b63d03f5f79c56e9e4120aec44ef,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_item_id,112650.0,NaN,NaN,NaN,1.197834,1.0,1.0,1.0,1.0,21.0,0.705124
product_id,112650,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,112650,3095,6560211a19b47992c3666cc44a7e94c0,2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
shipping_limit_date,112650,NaN,NaN,NaN,2018-01-07 15:36:52.192685,2016-09-19 00:15:34,2017-09-20 20:57:27.500000,2018-01-26 13:59:35,2018-05-10 14:34:00.750000,2020-04-09 22:35:08,NaN
price,112650.0,NaN,NaN,NaN,120.653739,0.85,39.9,74.99,134.9,6735.0,183.633928
freight_value,112650.0,NaN,NaN,NaN,19.99032,0.0,13.08,16.26,21.15,409.68,15.806405


### 4.5 Pedidos, productos y vendedores únicos

Se compara la cantidad de filas contra la cantidad de `order_id` distintos: si hay más
filas que pedidos, significa que un pedido puede ocupar varias filas.

In [19]:
print("Filas totales      :", f"{len(order_items_df):,}")
print("order_id únicos    :", f"{order_items_df['order_id'].nunique():,}")
print("product_id únicos  :", f"{order_items_df['product_id'].nunique():,}")
print("seller_id únicos   :", f"{order_items_df['seller_id'].nunique():,}")

Filas totales      : 112,650
order_id únicos    : 98,666
product_id únicos  : 32,951
seller_id únicos   : 3,095


### 4.6 Estadísticas de precios

`price` es el precio del ítem y `freight_value` el costo de envío asociado.

In [20]:
order_items_df[["price", "freight_value"]].describe().T

,count,mean,std,min,25%,50%,75%,max
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68


Percentiles de `price` para ver cómo se distribuye y detectar la cola de valores altos.

In [21]:
order_items_df["price"].quantile([0.01, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00])

0.01       9.99
0.25      39.90
0.50      74.99
0.75     134.90
0.95     349.90
0.99     890.00
1.00    6735.00
Name: price, dtype: float64

### 4.7 Pedidos con varios ítems

`order_item_id` es un contador secuencial dentro de cada pedido (1, 2, 3...). Primero se
mira cuántos ítems tiene cada pedido.

In [22]:
items_por_pedido = order_items_df.groupby("order_id").size()

print("Promedio de ítems por pedido:", round(items_por_pedido.mean(), 2))
print("Máximo de ítems en un pedido:", items_por_pedido.max())

items_por_pedido.value_counts().sort_index()

Promedio de ítems por pedido: 1.14
Máximo de ítems en un pedido: 21


1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

Algunos ejemplos concretos de pedidos con más de un ítem.

In [23]:
pedidos_multiples = items_por_pedido[items_por_pedido > 1].index[:3]

order_items_df[order_items_df["order_id"].isin(pedidos_multiples)].sort_values(
    ["order_id", "order_item_id"]
)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
13,0008288aa423d2a3f00fcb17cd7d8719,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
14,0008288aa423d2a3f00fcb17cd7d8719,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.10
42,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
43,001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63


---
## 5. Calidad de los datos

Antes de construir el pipeline conviene saber con qué problemas hay que convivir:
valores nulos, filas duplicadas y valores fuera de rango.

### 5.1 Valores nulos

In [24]:
pd.DataFrame(
    {
        "nulos": orders_df.isna().sum(),
        "porcentaje": (orders_df.isna().mean() * 100).round(2),
    }
)

,nulos,porcentaje
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [25]:
pd.DataFrame(
    {
        "nulos": order_items_df.isna().sum(),
        "porcentaje": (order_items_df.isna().mean() * 100).round(2),
    }
)

,nulos,porcentaje
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


### 5.2 Duplicados

En `orders_df` no debería repetirse `order_id`. En `order_items_df` la clave natural es
la combinación `(order_id, order_item_id)`.

In [26]:
print("orders_df - filas duplicadas            :", orders_df.duplicated().sum())
print("orders_df - order_id duplicados         :", orders_df["order_id"].duplicated().sum())
print("order_items_df - filas duplicadas       :", order_items_df.duplicated().sum())
print(
    "order_items_df - (order_id, order_item_id) duplicados:",
    order_items_df.duplicated(subset=["order_id", "order_item_id"]).sum(),
)

orders_df - filas duplicadas            : 0
orders_df - order_id duplicados         : 0
order_items_df - filas duplicadas       : 0
order_items_df - (order_id, order_item_id) duplicados: 0


### 5.3 Valores anómalos en columnas importantes

Se revisan nulos y valores imposibles en las columnas que va a usar el pipeline:
`order_id`, `product_id` y `price`.

In [27]:
print("order_id nulos        :", order_items_df["order_id"].isna().sum())
print("product_id nulos      :", order_items_df["product_id"].isna().sum())
print("price nulos           :", order_items_df["price"].isna().sum())
print("price <= 0            :", (order_items_df["price"] <= 0).sum())
print("freight_value < 0     :", (order_items_df["freight_value"] < 0).sum())

order_id nulos        : 0
product_id nulos      : 0
price nulos           : 0
price <= 0            : 0
freight_value < 0     : 0


Los ítems más caros, para verificar si son valores plausibles o errores de carga.

In [28]:
order_items_df.nlargest(5, "price")[
    ["order_id", "order_item_id", "product_id", "price", "freight_value"]
]

,order_id,order_item_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,6735.0,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,6729.0,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,6499.0,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,4799.0,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,4690.0,74.34


### 5.4 Integridad referencial entre las dos tablas

¿Todos los `order_id` del detalle existen en la cabecera? ¿Hay pedidos sin ítems?

In [29]:
ids_orders = set(orders_df["order_id"])
ids_items = set(order_items_df["order_id"])

print("order_id en items que NO están en orders:", len(ids_items - ids_orders))
print("order_id en orders SIN ítems asociados  :", len(ids_orders - ids_items))

order_id en items que NO están en orders: 0
order_id en orders SIN ítems asociados  : 775


Los pedidos sin ítems suelen concentrarse en determinados estados. Vale la pena mirarlo.

In [30]:
sin_items = orders_df[~orders_df["order_id"].isin(ids_items)]

print("Pedidos sin ítems:", len(sin_items))
sin_items["order_status"].value_counts()

Pedidos sin ítems: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

---
## 6. Relación entre `orders_df` y `order_items_df`

Ambas tablas se vinculan por `order_id`: es uno a muchos, un pedido de la cabecera puede
tener varias filas en el detalle.

### 6.1 Merge exploratorio

In [31]:
merged_df = orders_df.merge(order_items_df, on="order_id", how="inner")

print("orders_df      ->", orders_df.shape)
print("order_items_df ->", order_items_df.shape)
print("merged_df      ->", merged_df.shape)

orders_df      -> (99441, 8)
order_items_df -> (112650, 7)
merged_df      -> (112650, 14)


El merge devuelve tantas filas como ítems (no como pedidos): eso confirma la relación
uno a muchos.

### 6.2 Columnas relevantes del resultado

In [32]:
cols_relevantes = [
    "order_id",
    "order_purchase_timestamp",
    "order_status",
    "product_id",
    "order_item_id",
    "price",
]

merged_df[cols_relevantes].head(10)

,order_id,order_purchase_timestamp,order_status,product_id,order_item_id,price
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,delivered,87285b34884572647811a353c7ac498a,1,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,delivered,595fac2a385ac33a80bd5114aec74eb8,1,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,delivered,aa4383b373c6aca5d8797843e5594415,1,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,delivered,d0b61bfb1de832b15ba9d266ca96e5b0,1,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,delivered,65266b2da20d04dbe00c5c2d3bb7859e,1,19.90
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,delivered,060cb19345d90064d1015407193c233d,1,147.90
6,136cce7faa42fdb2cefd53fdc79a6098,2017-04-11 12:22:08,invoiced,a1804276d9941ac0733cfd409f5206eb,1,49.90
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:30,delivered,4520766ec412348b8d4caa5e8a18c464,1,59.99
8,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,delivered,ac1789e492dcd698c5c10b97a671243a,1,19.90
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:02,delivered,9a78fb9862b10749a117f7fc3c31f051,1,149.99


### 6.3 Verificación del cruce

Con `how="outer"` e `indicator=True` se ve de qué lado quedó cada fila y si hay pedidos
que no cruzan.

In [33]:
merge_check = orders_df.merge(order_items_df, on="order_id", how="outer", indicator=True)

merge_check["_merge"].value_counts()

_merge
both          112650
left_only        775
right_only         0
Name: count, dtype: int64

Un mismo pedido puede tener ítems de varios vendedores o productos distintos. Este
resumen por pedido lo deja a la vista.

In [34]:
resumen_pedido = merged_df.groupby("order_id").agg(
    items=("order_item_id", "count"),
    productos_distintos=("product_id", "nunique"),
    vendedores_distintos=("seller_id", "nunique"),
    total_precio=("price", "sum"),
)

resumen_pedido.sort_values("items", ascending=False).head(10)

,items,productos_distintos,vendedores_distintos,total_precio
order_id,,,,
8272b63d03f5f79c56e9e4120aec44ef,21,3,1,31.80
1b15974a0141d54e36626dca3fdc731a,20,1,1,2000.00
ab14fdcfbe524636d65ee38360e22ce8,20,1,1,1974.00
9ef13efd6949e4573a18964dd1bbe7f5,15,1,1,765.00
428a2f660dc84138d969ccd69a0ab6d5,15,1,1,982.35
9bdc4d4c71aa1de4606060929dee888c,14,1,1,419.86
73c8ab38f07dc94389065f7eba4f297a,14,1,1,826.00
37ee401157a3a0b28c9c6d0ed8c3b24b,13,1,1,389.87
af822dacd6f5cff7376413c03a388bb7,12,2,1,61.26


---
## 7. ¿Cómo se representa la cantidad de productos?

`order_items_df` **no tiene una columna `quantity`**. La hipótesis a verificar es que
cada fila representa **una unidad**: si alguien compra 3 unidades del mismo producto,
aparecen 3 filas con el mismo `order_id` y `product_id`, y `order_item_id` numerado
1, 2, 3.

### 7.1 Conteo de filas por `(order_id, product_id)`

In [35]:
cantidad_por_producto = (
    order_items_df.groupby(["order_id", "product_id"]).size().reset_index(name="cantidad")
)

cantidad_por_producto.head()

,order_id,product_id,cantidad
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,1
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,1
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,1
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,1


### 7.2 ¿Cuántos casos tienen el mismo producto repetido?

In [36]:
repetidos = cantidad_por_producto[cantidad_por_producto["cantidad"] > 1]

print("Combinaciones (order_id, product_id):", f"{len(cantidad_por_producto):,}")
print("Combinaciones con cantidad > 1      :", f"{len(repetidos):,}")
print(
    "Porcentaje                          :",
    round(len(repetidos) / len(cantidad_por_producto) * 100, 2),
    "%",
)

repetidos["cantidad"].value_counts().sort_index()

Combinaciones (order_id, product_id): 102,425
Combinaciones con cantidad > 1      : 7,088
Porcentaje                          : 6.92 %


cantidad
2     5382
3      953
4      390
5      168
6      172
7        4
8        2
9        2
10       5
11       1
12       2
13       1
14       2
15       2
20       2
Name: count, dtype: int64

### 7.3 Ejemplos concretos

Se toman los pedidos con más repeticiones para ver las filas completas.

In [37]:
ejemplos = repetidos.sort_values("cantidad", ascending=False).head(3)["order_id"]

order_items_df[order_items_df["order_id"].isin(ejemplos)].sort_values(
    ["order_id", "order_item_id"]
)[["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"]]

,order_id,order_item_id,product_id,seller_id,price,freight_value
11932,1b15974a0141d54e36626dca3fdc731a,1,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11933,1b15974a0141d54e36626dca3fdc731a,2,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11934,1b15974a0141d54e36626dca3fdc731a,3,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11935,1b15974a0141d54e36626dca3fdc731a,4,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11936,1b15974a0141d54e36626dca3fdc731a,5,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11937,1b15974a0141d54e36626dca3fdc731a,6,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11938,1b15974a0141d54e36626dca3fdc731a,7,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11939,1b15974a0141d54e36626dca3fdc731a,8,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11940,1b15974a0141d54e36626dca3fdc731a,9,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12
11941,1b15974a0141d54e36626dca3fdc731a,10,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,100.0,10.12


### 7.4 ¿El precio se repite en cada fila?

Si `price` es el precio unitario, todas las filas de un mismo `(order_id, product_id)`
deberían tener el mismo valor (`precios_distintos = 1`) y el total del producto sería
`precio_unitario * cantidad`.

In [38]:
resumen_precio = (
    order_items_df.groupby(["order_id", "product_id"])["price"]
    .agg(
        filas="count",
        precios_distintos="nunique",
        precio_min="min",
        precio_max="max",
        total="sum",
    )
    .reset_index()
)

repetidos_precio = resumen_precio[resumen_precio["filas"] > 1]

print("Casos con más de una fila            :", f"{len(repetidos_precio):,}")
print(
    "De esos, con un único precio unitario:",
    f"{(repetidos_precio['precios_distintos'] == 1).sum():,}",
)

repetidos_precio.head(10)

Casos con más de una fila            : 7,088
De esos, con un único precio unitario: 7,088


,order_id,product_id,filas,precios_distintos,precio_min,precio_max,total
13,0008288aa423d2a3f00fcb17cd7d8719,368c6c730842d78016ad823897a372db,2,1,49.90,49.90,99.80
31,00143d0f86d6fbd9f9b38ab440ac16f5,e95ee6822b66ac6058e2e4aff656071a,3,1,21.33,21.33,63.99
39,001ab0a7578dd66cd4b0a71f5b6e1e41,0b0172eb0fd18479d29c3bc122c058c2,3,1,24.89,24.89,74.67
43,001d8f0e34a38c37f7dba2a37d4eba8b,e67307ff0f15ade43fcb6e670be7a74c,2,1,18.99,18.99,37.98
69,002c9def9c9b951b1bec6d50753c9891,2d9ff06c8870a518f5f6909774e140fb,2,1,78.00,78.00,156.00
80,003324c70b19a16798817b2b3640e721,2b939dc9b176d7fa21594d588815d4a4,2,1,102.90,102.90,205.80
93,003822434f91204da0a51fe4cf2aba18,99e71b776debf2f01a69dce207e3e4f8,2,1,69.00,69.00,138.00
102,003f201cdd39cdd59b6447cff2195456,656e0eca68dcecf6a31b8ececfabe3e8,2,1,85.00,85.00,170.00
125,005059edee63c8c708ba61910793b31b,84f456958365164420cfc80fbe4c7fab,2,1,92.00,92.00,184.00
126,00526a9d4ebde463baee25f386963ddc,0c4a0f8ab44f9acd2d04e7024f9ba362,4,1,33.89,33.89,135.56


## 8. Conclusiones de la exploración

### Estructura

* `orders_df` tiene **99.441 pedidos y 8 columnas**. `order_id` es único en todos los registros.
* `customer_id` también es único por pedido, por lo que **no permite identificar compradores recurrentes**. Para eso se necesita `customer_unique_id`, disponible en otro archivo.
* `order_items_df` tiene **112.650 filas**, correspondientes a **98.666 pedidos, 32.951 productos y 3.095 vendedores**. La clave es `(order_id, order_item_id)`.
* La diferencia entre pedidos e ítems muestra una relación **uno a muchos**. Cada pedido tiene en promedio **1,14 ítems**, con un máximo de 21.

### Fechas y estados

* Las fechas están almacenadas como texto, por lo que deben convertirse antes de trabajar con ellas.
* Los pedidos abarcan desde el **04/09/2016 hasta el 17/10/2018**, aunque los extremos tienen muy pocos registros y no son representativos.
* `delivered` concentra el **97,02 % de los pedidos**. Los demás estados tienen una participación mucho menor, por lo que será necesario definir cuáles se consideran en cada métrica.

### Calidad de los datos

* No se encontraron duplicados ni claves repetidas en ninguna de las dos tablas.
* Los valores nulos aparecen únicamente en fechas de `orders_df` y están relacionados principalmente con pedidos que todavía no llegaron a determinadas etapas del proceso.
* `order_items_df` no tiene valores nulos.
* No se detectaron precios ni costos de envío con valores imposibles. Los precios presentan algunos valores altos, por lo que conviene considerar su distribución antes de utilizar promedios.
* `shipping_limit_date` presenta algunos registros con fechas fuera del período analizado, llegando hasta abril de 2020. Es un dato que debería revisarse o documentarse.
* Hay **775 pedidos sin ítems asociados**, principalmente pedidos `unavailable` y `canceled`. Esto debe considerarse al definir los joins y las métricas.

### Relación entre las tablas

* Al relacionar ambas tablas mediante `order_id`, se obtienen las **112.650 filas de `order_items_df`**, sin registros de ítems que no tengan un pedido correspondiente.
* Un pedido puede contener varios productos y vendedores, por lo que las métricas a nivel pedido deben calcularse después de realizar la agregación correspondiente.

### Granularidad

* `order_items_df` no tiene una columna de cantidad. **Cada fila representa una unidad del producto vendido**.
* La cantidad se obtiene contando las filas agrupadas por `order_id` y `product_id`.
* De las **102.425 combinaciones** entre pedido y producto, **7.088 (6,92 %) tienen más de una unidad**.
* En esos casos, el precio se mantiene igual entre las filas, por lo que `price` representa el **precio unitario**. El importe total se obtiene multiplicando precio por cantidad, o simplemente sumando `price`.
* El **90 % de los pedidos tiene un solo ítem**, por lo que esta diferencia puede pasar desapercibida si se analiza el dataset directamente a nivel de fila.

### Próximo paso

La transformación puede partir de `order_items`, agrupando por `(order_id, product_id)` para obtener la cantidad y el importe, y luego incorporando desde `orders` la fecha y el estado del pedido.

Antes de construir el pipeline quedan dos decisiones principales: **qué estados incluir en las métricas y cómo tratar los 775 pedidos sin ítems**.

### Propuesta

La transformación va a comenzar desde order_items. Como cada fila representa una unidad vendida, se agruparán los datos por (order_id, product_id) para calcular la cantidad de unidades y el importe total vendido de cada producto dentro de cada pedido.

Después, ese resultado se relacionará con orders mediante order_id para incorporar información del pedido, principalmente la fecha de compra y el estado.

Para la capa Silver se conservarán todos los estados de pedido. El filtrado de ventas válidas, por ejemplo considerar únicamente pedidos delivered, se realizará más adelante en la capa Gold con dbt.

Los 775 pedidos sin ítems se mantendrán en la tabla Silver de orders, porque siguen siendo pedidos válidos dentro de la fuente. No aparecerán en la tabla de ventas por producto, ya que no tienen ningún ítem asociado.